# Burn Simulation

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np


import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.express.colors import qualitative as pallettes

from copy import deepcopy

In [ ]:
def profile2(theta):
    r = 10 / (np.cos(np.pi/4 - (theta % (np.pi/2))))
    return r

In [ ]:
def profile1(theta):
    a = 10 # offset
    b = 5 # amplitude
    c = 7 # number of lobes
    r = a + b * np.cos(c*theta)
    return r

In [ ]:
# Convert between Cartesian/Polar coordinates

def cart2pol(x, y):
    rho = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return(rho, phi)

def pol2cart(rho, phi):
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return(x, y)

def magn(x, y):
    # magnitude of vector
    return np.sqrt(x**2 + y**2)

def rad2deg(rad):
    # convert radians to degrees
    return rad * 180 / np.pi

def circumference(X,Y):
    with np.errstate(divide='ignore', invalid='ignore'):
        dx = np.diff(X)
        dy = np.diff(Y)
    dxdy = np.sqrt(dx**2 + dy**2)
    C = np.sum(dxdy)
    return C 

In [ ]:
def normals(X, Y, ):
    # Calculates normals of the curve

    Gy = np.gradient(Y, X)

    #Check if any gradients got inf or nan
    #assert sum(np.isnan(Gy)) == 0

    Gx = np.ones_like(Gy) # not sure this is correct btw.
    # Gx are all 1s assumes the curve Y is a function of X, which is an axis with uniform spacing.
    # But in our case, the X and Y are both functions of theta, since they are generated from a polar coordinate system,
    # So, the assumption may not hold, and the normals may not be accurate. 
    # And that is probably why in some regions the normals don't look perpendicular to the curve in the plot.

    # Calculate normals
    L = magn(Gx, Gy)
    Nx, Ny = -Gy/L, Gx/L 
    xy  = np.array([X,Y])
    NxNy = np.array([Nx,Ny])

    # Ensure all normal vectors point in the same 
    # direction (Relative to origin) 
    dot_products = np.sum(xy * NxNy, axis=0)

    scaling_factor_S = np.sign(dot_products)

    S_reshaped = scaling_factor_S[np.newaxis, :]

    N_consistent = NxNy * S_reshaped 
    Nx, Ny =  N_consistent
    return Nx, Ny

In [ ]:
#simulation points
n = 1000 

# Theta (radians). To avoid /div0 start from 1 
T = np.linspace(1, np.pi*2 +1 , n)

# Theta degrees
TD = rad2deg(T)

In [ ]:
# Calculate Radius for each Theta
R = profile1(T)

## Working in cartesian


In [ ]:
X, Y = pol2cart(R, T)

Calculate the gradient (which in 1d case is just derivative)

In [ ]:
Gy = np.gradient(Y, X)

Check if any gradients got inf or nan

In [ ]:
sum(np.isnan(Gy))

In [ ]:
infs = np.argwhere(Gy == -np.inf)
infs

In [ ]:
infs = np.argwhere(Gy == np.inf)
infs

Optionally: set them to 0? 

In [ ]:
Gy[infs] = 0

In [ ]:
px.line(Gy)

In [ ]:
Gx = np.ones_like(Gy)

In [ ]:
L = magn(Gx, Gy)
L.shape


Calculate normals

In [ ]:
Nx, Ny = -Gy/L, Gx/L 

In [ ]:
xy  = np.array([X,Y])

In [ ]:
NxNy = np.array([Nx,Ny])

In [ ]:
NxNy.shape

Ensure all normal vectors point in the same direction 
(Relative to origin) 

In [ ]:
dot_products = np.sum(xy * NxNy, axis=0)
dot_products.shape

In [ ]:
scaling_factor_S = np.sign(dot_products)

# 3. Reshape S to be an N x 1 array to multiply correctly with N_initial (N x 2)
#    np.newaxis expands the dimension: [s1, s2, ...] -> [[s1], [s2], ...]
S_reshaped = scaling_factor_S[np.newaxis, :]

In [ ]:
S_reshaped.shape

In [ ]:
N_consistent = NxNy * S_reshaped 
N_consistent.shape

In [ ]:
Nx, Ny =  N_consistent

In [ ]:
Nx.shape

Calculate the new points after iteration

In [ ]:
# Iteration step size
d = -.1

X1, Y1 =  X - d * Nx , Y - d * Ny

In [ ]:
# region of interest
roi = {'x':(-16,16), 'y':(-16,16)}
#roi = {'x':(0.5, 2.), 'y':(4.5, 6.5)} # for the 7-lobe shape
print(f"zooming to x: {roi['x']}, y: {roi['y']}")

In [ ]:

fig = px.line(x = X, y = Y,width=1000, height=1000)
fig.add_trace(go.Scatter(x=X1 , y=Y1, mode='lines', name='next step'))
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
fig.show()

Calcualte circumference

In [ ]:


circumference(X,Y)

## Problem

In [ ]:
# Larger iteration step size
d = .3

# Offset
Ox, Oy =  d * Nx , d * Ny
# End
X1, Y1 =  X + Ox,  Y + Oy 

In [ ]:

fig = px.line(x = X, y = Y,width=800, height=800, )
fig.add_trace(go.Scatter(x=X1 , y=Y1, mode='lines', name='next step'))
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
fig.show()

If I increase iteration step size, then in sharp corners of the shape we'll inevitably get normals crossing to the other side of the shape, thereby creating these "swallow tail" patterns.  
Same goes for any further iterations.  
Naturally, circumference calculation is also affected by this.

I need to cut these tails off.

## Vector intersections

Observing the plot, I see that the normals producing the tail are the ones that intersect. These are the ones I need to remove



In [ ]:
fig = ff.create_quiver(X,Y, Ox,Oy,
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.05,
                       name='offset',
                       line_width=3, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=X, y=Y,mode='lines', name='current'))
fig.add_trace(go.Scatter (x=X1, y=Y1,mode='lines',  name='next step'))

fig.update_layout(  width=800, height=800)
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
# Add points to figure
fig.show()

Solution Approach:  
Drop all normals that are adjacent and intersect within distance d (the step size) from their origin.  


**Summary of the function:**  

Given: an array of size n of 2d vectors. Each vector has an origin X,Y and direction Nx, Ny.  
Order of vectors matters. For each pair of vectors adjacent within the array, find their intersection point.   
No need to check for intersections of all possible pairs of vectors. Just the adjacent ones.  
Only intersections in the positive direction of vectors matter.  

In [ ]:
def adjacent_intersections(X, Y, Nx, Ny, forward_only=True, eps=1e-12):
    """Compute intersections for adjacent 2D rays (vectorized)."""
    X = np.asarray(X).ravel()
    Y = np.asarray(Y).ravel()
    Nx = np.asarray(Nx).ravel()
    Ny = np.asarray(Ny).ravel()
    if not (X.size == Y.size == Nx.size == Ny.size):
        raise ValueError('All inputs must have same length')
    # Stack as (n,2) arrays and form adjacent pairs (n-1)
    c = np.stack((X, Y), axis=1)  # (n,2)
    v = np.stack((Nx, Ny), axis=1)
    c_i = c[:-1]; c_j = c[1:]
    v_i = v[:-1]; v_j = v[1:]
    C = c_j - c_i
    # 2D cross product (scalar) for arrays of shape (m,2)
    def cross2(a, b):
        return a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
    den = cross2(v_i, v_j)
    num_ti = cross2(C, v_j)
    num_tj = cross2(C, v_i)
    # safe division with NumPy (will produce inf/nan where den==0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ti = num_ti / den
        tj = num_tj / den
    # validity mask: non-parallel and finite
    valid = np.isfinite(den) & (np.abs(den) > eps)

    if forward_only:
        valid &= (ti > 0) & (tj > 0) 

    # Intersection points computed from ray i: p = c_i + ti * v_i
    Px = c_i[:, 0] + ti * v_i[:, 0]
    Py = c_i[:, 1] + ti * v_i[:, 1]
    # Mask invalid entries as NaN for clarity
    invalid = ~valid
    if invalid.any():
        Px = Px.astype(float)
        Py = Py.astype(float)
        ti = ti.astype(float)
        tj = tj.astype(float)
        # Px[invalid] = np.nan
        # Py[invalid] = np.nan
        # ti[invalid] = np.nan
        # tj[invalid] = np.nan
    return Px, Py, ti, tj, valid


Px, Py, ti, tj, valid = adjacent_intersections(X, Y, Nx, Ny, forward_only=True)

In [ ]:
# filter which drops points that fall within d from previous line

def myfilter(ti,tj,d):
    a= np.append(ti, np.array(True))
    b = np.append(np.array(True), tj)
    flt = (a <= 2.2*np.abs(d) ) & (b <= 2.2*np.abs(d) )
    return flt

In [ ]:
filt = myfilter(ti,tj,d) & np.append(valid, False)

Xn = X1[~filt]
Yn = Y1[~filt]

In [ ]:
fig = ff.create_quiver(X,Y, Ox, Oy, 
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.05,
                       name='offset',
                       line_width=3, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=X, y=Y,mode='lines', name='current step'))
fig.add_trace(go.Scatter (x=Px, y=Py,mode='markers',  name='intersections'))
fig.add_trace(go.Scatter (x=Xn, y=Yn,mode='lines',  name='next step'))

fig.update_layout(  width=800, height=800)
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
# Add points to figure
fig.show()

There is some progress: The vectors situated in the sharpest part of the curve got filtered out.  

However intersecting vectors still remain. 
Those are the vectors that fail our initial assumption that only adjacent intersecting vectors need to be dropped.  

The approach needs to be modified.  


## Modified approach

It is necessary to check for intersections of not only the adjacent vectors, but more.  
Thankfully, no need to check ALL the possible intersections, but a relatively small amount of vectors  
within some sliding window. This will keep the performance sane.

Generally the procedure should be:  
Run a window over the array, with defined length and offset.  
For each offset, only check intersections of the first vector vs. all the rest of the vectors inside the window.  
All the normals pairs that have their intersections within the bounds of both normals, are to be dropped.  

The implementation of this approach is below:


In [ ]:
def window_intersections(X, Y, Nx, Ny, window_size=11, step=1, tol=0.1, forward_only=True, eps=1e-12):
    """Find intersections within sliding windows.
    For each window start, test the first ray (index i=start) against all following rays in the window.
    Return compact arrays of hits (i, j, ti, tj, Px, Py)."""

    X = np.asarray(X).ravel()
    Y = np.asarray(Y).ravel()
    Nx = np.asarray(Nx).ravel()
    Ny = np.asarray(Ny).ravel()
    if not (X.size == Y.size == Nx.size == Ny.size):
        raise ValueError('All inputs must have same length')
    n = X.size
    if window_size < 2:
        raise ValueError('window_size must be >= 2')
    # Prepare arrays
    c = np.stack((X, Y), axis=1)  # (n,2)
    v = np.stack((Nx, Ny), axis=1)

    hits_i = []
    hits_j = []
    hits_ti = []
    hits_tj = []
    hits_P = []
    hits_Py = []
    classes = []
    condis = []
    condjs = []
    filt = np.zeros(n).astype(bool)
    iis = []

    # Helper cross product for arrays
    # def cross2_arr(a_x, a_y, b_x, b_y):
    #     return a_x * b_y - a_y * b_x
    # Slide window (simple Python loop over windows; per-window ops are vectorized)

    print("points to do:",  n - window_size + 1)
    
    for i in range(0, n - window_size + 1, step):
        this = i + int(window_size/2)
        c0 = c[this]            # (2,)
        v0 = v[this]            # (2,)
        c_block = c[i+1:i+window_size]    # (m,2)
        v_block = v[i+1:i+window_size]    # (m,2)
        # Cx = c_block[:,0] - c0[0]
        # Cy = c_block[:,1] - c0[1]
        C = c_block - c0
        # den = cross(v0, v_block)
        # den = cross2_arr(v0[0], v0[1], v_block[:,0], v_block[:,1])
        den = np.cross(v0,v_block)
        # numerators
        # num_ti = cross2_arr(Cx, Cy, v_block[:,0], v_block[:,1])
        num_ti = np.cross(C, v_block)
        # num_tj = cross2_arr(Cx, Cy, v0[0], v0[1])
        num_tj = np.cross(C, v0)

        with np.errstate(divide='ignore', invalid='ignore'):
            ti = num_ti / den
            tj = num_tj / den

        valid = den != 0 #& (np.abs(den) > eps)
        valid = valid & (((0 < ti) & (ti  < tol*3)) | ((0 < tj ) & (tj < tol*3)))

        condi = (0 < ti) & (ti < tol*.9)
        condj = (0 < tj) & (tj < tol)


        # these must be none true
        outer = all( condi== False) #| all( condj == False )

        clas = valid*1 + condi*1 + condj*1

            
        # filt[this] = (not any((condi) & (condi != condj))) #| outer
        
        filt[this] = (not any((condi) & (condi != condj))) & (not all( condi== False))

        #filt[this] = outer
        #valid &= cond
        # if not np.any(valid):
        #     continue

        # if filt.sum() > 0:
        #     print('wait', i, end='')

        # compute intersection points for valid entries
        # vi_x = v0[0]; 
        # vi_y = v0[1]
        # Px = c0[0] + ti * vi_x
        # Py = c0[1] + ti * vi_y

        P = c0 + np.stack((ti,ti), axis=1) * v0

        # append hits
        # for valid in np.nonzero(valid)[0]:
        # hits_i.append(i)
        # hits_j.append(i + 1 + int(idx_local))

        iis.append(i)
        hits_ti.append(ti[valid])
        hits_tj.append(tj[valid])
        hits_P.append(P[valid])
        classes.append(clas[valid])
        condis.append(condi[valid])
        condjs.append(condj[valid])

    res = {
            "iis":     np.array(iis),
            # hits_i: hits_i,
            # hits_j: hits_j,
            "hits_ti": np.concatenate(hits_ti), 
            "hits_tj": np.concatenate(hits_tj), 
            "hits_P":  np.concatenate(hits_P), 
            "classes": np.concatenate(classes), 
            "condis":  np.concatenate(condis), 
            "condjs":  np.concatenate(condjs), 
            "filt":    np.array(filt),
        }
    return res

    # if len(hits_i) == 0:
    #     return (np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=float),
    #             np.array([], dtype=float), np.array([], dtype=float), np.array([], dtype=float),
    #             np.array([]), np.array([]), np.array([]), np.array([]), np.array([]) )

    # return (np.array(hits_i, dtype=int), np.array(hits_j, dtype=int), np.array(hits_ti, dtype=float),
    #         np.array(hits_tj, dtype=float), np.array(hits_P, dtype=float), np.array(hits_Py, dtype=float),
    #         np.array(classes), np.array(condis), np.array(condjs), np.array(filt), np.array(iis) )

In [ ]:

def step(I, X, Y, d, s):
    Nx, Ny = normals(X, Y)

    Ex, Ey =  X + d * Nx , Y + d * Ny

    res = window_intersections(X, Y, Nx, Ny, window_size=100, step=1, tol=d) # *(1+s*0.1)

    iis, ti_w, tj_w, P, clss, condi, condj, filt = res.values()
    # filt is True where the points should be filtered out / dropped
    
    #Px, Py, ti, tj, valid = adjacent_intersections(Ex, Ey, Nx, Ny, forward_only=True)
    #filt =  np.concatenate([i_idx , j_idx])  #myfilter(ti,tj,d) # & np.append(valid, False)
    
    # if filt.size != 0:
    #     Ex[filt] = np.nan
    #     Ey[filt] = np.nan

    if filt.size != 0:
        Ex[filt] = np.nan
        Ey[filt] = np.nan



        X1 = Ex[~filt]
        Y1 = Ey[~filt]
        I1 =  I[~filt]


    res = { 'I':I,
        'step': s,
        # 'Theta': T, 
        # 'ThetaDeg':TD, 
        # 'Radius': R, 
        # 'Gradient': GP,
        'X': X,
        'Y': Y,
        'Nx': Nx,
        'Ny': Ny,
        'Ex': Ex,
        'Ey': Ey,
        'I1': I1,
        'X1': X1,
        'Y1': Y1,
        'filt': filt,
        'circ': circumference(X,Y),
        'iis': iis
        }
    extra =  { 'step': s,
        'Px': P[:,0],
        'Py': P[:,1],
        'ti': ti_w,
        'tj': tj_w,
        'condi': condi, 
        'condj': condj,
        'clss' : clss
        # 'valid': np.append(valid, False),
        }


    return I1, X1, Y1, res, extra


In [ ]:
I1, X1, Y1, res, extra =  step(**subst)

In [ ]:
tmp = res
tmp.update(extra)
# [print(k, v.shape) for k,v in tmp.items() if type(v) not in (int, float, np.float64)]
#tmp = {k: tmp[k] for k in "I, X, Y, Nx, Ny, X1, Y1, Ex, Ey, Px, Py".split(", ")}
describe(tmp)
#[print(k, v.shape) for k,v in tmp.items()]

In [ ]:

dots_and_arrows(**tmp, d = d)

In [ ]:

def dots_and_arrows(I, X, Y, Nx, Ny, I1, X1, Y1, Ex, Ey, Px, Py, d, clss, filt, **kwargs):
    fig = ff.create_quiver(X, Y, +d * Nx, +d * Ny, scale=1, arrow_scale=.05, name='offset',hovertext=I)
    fig.add_trace(go.Scatter(x=Px, y=Py, mode='markers', 
                             marker=dict(size=6, color = clss*1, symbol = 'x') , 
                             hovertext = clss,
                             name='window_hits'))
    fig.add_trace(go.Scatter(x=X1, y=Y1, mode='lines', marker=dict(size=6, color=filt*1), 
                             hovertext=I1, name='X1Y1'))
    fig.add_trace(go.Scatter(x=Ex, y=Ey, mode='lines', marker=dict(size=6, color=filt*1), 
                             hovertext=I1, name='ExEy'))
    fig.update_layout(width=800, height=800)
    # fig.update_xaxes(range=roi['x'])
    # fig.update_yaxes(range=roi['y'])
    fig.show()


#dots_and_arrows(I, X,Y, Nx, Ny, Ex, Ey, Px_w, Py_w,d)


In [ ]:
from Superformula.Formulas import formula1, formula2

In [ ]:
# args = ('superformula', 3, 1, 0.7, 2, 0, 1, 0)

# args = ('superformula', 5, 1, -0.38, 0.38, 0, 1, 0)

# args = ('superformula', 5, 1, -0.66, 0.95, 0, 1, 1)

args = ('trapez wave', 5, 1, -0.66, 0.95, 0, 1, 1)

In [ ]:
profile = formula2(*args)

In [ ]:
d = .11
steps = 20
n = 1000 

T = np.linspace(0, np.pi*2 , n)
I = np.arange(len(T))
# Calculate Radius for each Theta
R = profile(T)

X, Y = pol2cart(R, T)
Nx, Ny = normals(X, Y)
X1, Y1 =  X + d * Nx , Y + d * Ny


In [ ]:

def arc_lens(X,Y):
    """
    Calculate lengths of curve segments defined by points (X,Y).

    Args:
        X (_type_): _description_
        Y (_type_): _description_

    Returns:
        _type_: _description_
    """

    XY = np.stack((X,Y), axis=1)
    XY = np.concatenate((XY,XY[0:1,:]), axis=0)

    diff = np.diff(XY, axis=0)
    Lens = np.sqrt(np.sum(diff ** 2, axis=1))
    return Lens

In [ ]:
Lens = arc_lens(X,Y)
np.mean(Lens)

In [ ]:
px.line(Lens)

In [ ]:
# Demo: run window_intersections on current arrays and plot hits

#indices = slice(600,1000)

indices = slice(None)

subst = [ii[indices] for ii in [X, Y, Nx, Ny,]]

vrs = "I, X, Y, d, s"

subst = clip(vrs, indices)


In [ ]:


i_idx, j_idx, ti_w, tj_w, Px, Py, clss, condi, condj, filt, iiss = window_intersections(*subst, forward_only=False, window_size=90, step=1, tol=d)
print('window_intersections found', i_idx.size, 'hits')
if i_idx.size > 0:
    print('sample (i,j,ti,tj):')
    for a,b,ta,tb in zip(i_idx[:10], j_idx[:10], ti_w[:10], tj_w[:10]):
        print(a, b, round(ta,3), round(tb,3))
# Plot hits on top of the offset quiver





In [ ]:
s = 0

isSubscriptable = lambda obj: hasattr(obj, '__getitem__')

def describe(subst):
    fmt = "{n} {s}: {t}"
    summ = [fmt.format(n=k, t=type(v), s= v.shape if isSubscriptable(v) else v) for k,v in subst.items() ]
    print(*summ, sep='\n')

def clip(vars, indcs = slice(None)):

    rooster = vars.split(", ")
    #print(rooster)
    thevars = {k: globals()[k] for k in rooster}

    subst = {k: v[indcs] if isSubscriptable(v) else v  for k, v in thevars.items()}
    describe(subst)
    return subst






In [ ]:
describe(res)

In [ ]:
dots_and_arrows(I[indices], *subst, Ex_f[~filt], Ey_f[~filt], Px, Py,d, condi == condj)

In [ ]:
dots_and_arrows(*subst, Px, Py,d)

# Full simulation 

Following is all above logic summarized in easy to use functions.

In [ ]:



def run(func, d = .011, steps = 1, n = 1000 ):
    """runs the simulation

    Args:
        func (def): function defining the curve. must be of form Radius = f(Theta)
        d (float, optional): step size. Defaults to -.3.
        steps (int, optional): simulation steps. Defaults to 1.
        n (int, optional): simulation points. Defaults to 1000.

    Returns:
        dict: dictionary of results data
    """
    
    print('Simulation step size(d):',d)
    print('Simulation Steps:', steps)
    print('Curve points (n):', n)

    # Theta (radians)
    T = np.linspace(0, np.pi*2 , n)
    I = np.arange(len(T))
    # Calculate Radius for each Theta
    R = func(T)

    X, Y = pol2cart(R, T)

    data = []
    extras = []

    for s in range(steps):
        print("step:", s, "points:", X.shape)

        I, X, Y,res, extra = step(I, X, Y, d, s)

        data.append(deepcopy(res))
        extras.append(deepcopy(extra))


    return data, extras


## Run the simulation

In [ ]:
from Superformula.Formulas import formula1, formula2

In [ ]:
# args = ('superformula', 3, 1, 0.7, 2, 0, 1, 0)

# args = ('superformula', 5, 1, -0.38, 0.38, 0, 1, 0)

#args = ('superformula', 5, 1, -0.66, 0.95, 0, 1, 1)

args = ('trapez wave', 5, 1, -0.66, 0.95, 0, 1, 1)

In [ ]:
profile = formula2(*args)

In [ ]:

d = .011
steps = 80

n = 500

# Theta (radians). To avoid /div0 start from 1 
T = np.linspace(0, np.pi*2 , n)

R = profile(T)

Rmax = np.max(R) 


X, Y = pol2cart(R, T)
mode='lines'
fig = px.line(x=X, y=Y)
fig.update_layout(width=500 , height = 500, yaxis_range = [-Rmax, Rmax], xaxis_range = [-Rmax, Rmax])

fig.show()

In [ ]:


data, extra = run(profile, d, steps, n)

In [ ]:
describe(data[1])

In [ ]:
def pick(dt):
    fields = "I, step, X, Y, Nx, Ny, circ".split(', ')
    toplot = {f: dt[f] for f in fields}
    return toplot

pick(data[1])

In [ ]:
df = [pd.DataFrame(pick(dt)) for dt in data]
df = pd.concat(df, ignore_index=True)
df

In [ ]:
df.isna().sum()

In [ ]:
df[df.Nx.isna() | df.Ny.isna()]

In [ ]:
df = df.drop("circ", axis=1).dropna(axis=0)

In [ ]:
df

In [ ]:

df[['R','T']] = df.apply(lambda r: cart2pol(r['X'], r['Y']), axis=1, result_type='expand')
df['TD']      = df.apply(lambda r: rad2deg(r['T']), axis=1)

df['Step']    = df.apply(lambda r: f"{r.step}",  axis = 1) #  Circumference: {r.circ}",
df['face']    = df.apply(lambda r: f"i={r.I} | {round(r.TD)}°", axis = 1)

In [ ]:
df

## Interactive Polar plot

In [ ]:
Rub = np.ceil(df.R.max()) + 1


In [ ]:

fig = px.line_polar(df, r="R", theta="TD", line_close=True,
                    range_r=[0,Rub], animation_frame="Step", 
                    direction= "counterclockwise", start_angle=0,
                    #color_discrete_sequence=px.colors.sequential.Plasma_r, 
                    #template="plotly_dark",)
                    width=600, height=600
                    )
fig.show()

In [ ]:
px.line(y = df.circ,x = df.step, width=600, height=600)

## More detailed plot

In [ ]:
def paint(id, palt):
    n = int(id%len(palt))
    return palt[n]

In [ ]:
extras = [pd.DataFrame(d) for d in extra]
extras = pd.concat(extras, ignore_index=True)
extras['lbl']     = extras.apply(lambda r: paint(r.step,pallettes.Prism), axis=1)
extras

In [ ]:
def decimate(df, upper_lim = 20000, factor = None):
    # randomly sample len(df)/factor data points from.
    # useful for plotting results with high amount of points 
    if df.shape[0] > upper_lim:
        if factor is None:
            factor = int(df.shape[0] / upper_lim) + 1
        print("df length:", df.shape[0] , "> upper limit:", upper_lim, "decimating by factor:", factor)

        import random
        rids = random.sample(sorted(df.index.values), int(df.shape[0]/factor))
        rids.sort()
        sset = df.loc[rids,:]
        return sset.reset_index()
    
    return df

dec_extras = decimate(extras)


In [ ]:
# plot only specific steps 

stepsToPlot = [2,3] #[2,3,4] # [8,9,10] #

slc = np.isin(df.step, stepsToPlot ) # range(10)

# plot only region of interest
roi = {'x':(None,None), 'y':(None,None)} # auto
#roi = {'x':(-16,16), 'y':(-16,16)} # whole shape
#roi = {'x':(0.5, 2.), 'y':(4.5, 6.5)} # one corner
#roi = {'x':(-2.5 , -4.0), 'y':(3.5 , 4.5)} # other corner


cols ="I, step, X, Y, Nx, Ny, face".split(', ')

# print(k, v.shape ), Px, Py, ti, tj, lbl

i, s, Xx, Yy, Nx, Ny, face = df.loc[slc,cols].T.values

In [ ]:
slc = np.isin(extras.step,  stepsToPlot) # range(10)

cols = "Px, Py, ti, tj, step, lbl".split(', ')

Px, Py, ti, tj, s, lbl = extras.loc[slc,cols].T.values

In [ ]:
fig = ff.create_quiver(Xx, Yy,  d * Nx,  d * Ny,
#fig = ff.create_quiver(X, Y,  Nx,  Ny,
                        #x, y, u, v,
                       scale=1, 
                       arrow_scale=.05,
                       name='offset',
                       line_width=2, 
                       angle=np.pi/6,
                     #  width=800, height=800)
)

fig.add_trace(go.Scatter (x=Xx, y=Yy,mode='lines', name='curves', text = face))
fig.add_trace(go.Scatter (x=Px, y=Py,mode='markers',
                           marker_color=lbl, 
                           name='intersections'))

fig.update_layout( width=800, height=800)
fig.update_xaxes(range=roi['x'])
fig.update_yaxes(range=roi['y'])
# Add points to figure
fig.show()

In [ ]:
res = df.loc[slc].to_dict()
extraslc = np.isin(extras.step,  stepsToPlot) # range(10)

extra = extras.loc[extraslc].to_dict()

In [ ]:
tmp

In [ ]:
tmp = res
tmp.update(extra)
dots_and_arrows(**tmp, d = d)

The new approach proves to be much more robust.

Though problems still exist. 
As I cut the intersecting points, the sampling resolution in the corners decreases, making the line more jagged.
This adversely affects the calculation of normals at next step, specifically in the corners, where accuracy is most crucial.
As a result, some normals may diverge significantly from proper direction, becoming a seed for a growing "swallow tail" artefact down the line.  

For now, an increase in sim resolution and shorter step can deal with this problem, but more robust methods should be used in the future.  

For example:
- Perform curve fitting and resampling of the corners points after each cut.
- Use some of the closest intersection points in the resulting curve.
- Transfer some of the corner points of original curve to the next one, like changing a hat.

# End